### 1. Load Dependencies

In [ ]:
from pathlib import Path
import nltk

NLTK_DATA_DIR = Path("data/nltk_data")
NLTK_DATA_DIR.mkdir(parents=True, exist_ok=True)

nltk.data.path.append(str(NLTK_DATA_DIR))

nltk.download("punkt", download_dir=str(NLTK_DATA_DIR))
nltk.download("stopwords", download_dir=str(NLTK_DATA_DIR))
nltk.download("wordnet", download_dir=str(NLTK_DATA_DIR))
nltk.download("omw-1.4", download_dir=str(NLTK_DATA_DIR))

### 2.Import Dataset and Explore

In [ ]:
import pandas as pd
import sklearn
from sentence_transformers import SentenceTransformer

print("Setup successful!")

In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILE = Path("data/postings.csv")

df = pd.read_csv(DATA_FILE)

print("Using dataset:", DATA_FILE)
print("Shape:", df.shape)

In [ ]:
df.info()

In [ ]:
useful_cols = [
    "job_id",
    "company_name",
    "title",
    "description",
    "location",
    "formatted_work_type",
    "formatted_experience_level",
    "skills_desc",
    "work_type",
    "normalized_salary",
    "job_posting_url"
]

df = df[useful_cols]

df.head()

In [ ]:
df = df.drop_duplicates(subset=["title", "description"])

df = df.dropna(subset=["title", "description"])

print(df.shape)

In [ ]:
df = df.dropna(
    subset=[
        "title",
        "description"
    ]
)

print("After dropping empty title/description:", df.shape)

In [ ]:
df["company_name"] = df["company_name"].fillna("Unknown")

df["formatted_experience_level"] = (
    df["formatted_experience_level"]
    .fillna("Not Specified")
)

df["normalized_salary"] = (
    df["normalized_salary"]
    .fillna(-1)
)

In [ ]:
df = df.drop(columns=["skills_desc"])

In [ ]:
df["job_text"] = (
    df["title"].astype(str)
    + " "
    + df["description"].astype(str)
)

In [ ]:
print(df["job_text"].isnull().sum())

In [ ]:
df = df.sample(
    n=10000,
    random_state=42
).reset_index(drop=True)

In [ ]:
print(df.shape)

In [ ]:
from pathlib import Path

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(
    PROCESSED_DIR / "linkedin_jobs_clean.csv",
    index=False
)

print("Saved linkedin_jobs_clean.csv")

In [ ]:
df = pd.read_csv("linkedin_jobs_clean.csv")
df.head()

### 3. Preprocessing

In [ ]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [ ]:
#preprocess
def preprocess_text(text):
    # lowercase
    text = str(text).lower()

    # remove urls
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove punctuation and numbers
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # tokenize
    tokens = word_tokenize(text)

    # remove stopwords and short words
    tokens = [
        word for word in tokens
        if word not in stop_words and len(word) > 2
    ]

    # lemmatize
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)

In [ ]:
df["clean_job_text"] = df["job_text"].apply(preprocess_text)

In [ ]:
print("ORIGINAL:")
print(df.loc[0, "job_text"])

print("\n" + "="*80 + "\n")

print("CLEANED:")
print(df.loc[0, "clean_job_text"])

In [ ]:
df = df[
    df["clean_job_text"].str.strip() != ""
].reset_index(drop=True)

print(df.shape)

In [ ]:

df.to_csv(
    PROCESSED_DIR / "linkedin_jobs_preprocessed.csv",
    index=False
)
print("Saved linkedin_jobs_preprocessed.csv")


### 4. TF-IDF Baseline recommender

In [ ]:
# Load preprocessed data
import pandas as pd

df = pd.read_csv("linkedin_jobs_preprocessed.csv")

print(df.shape)
df[["title", "clean_job_text"]].head()

In [ ]:
# 2. Create tf-idf matrix
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df["clean_job_text"])

print(tfidf_matrix.shape)

In [ ]:
# 3. Create resume preprocessing
resume_text = """
Experienced data analyst skilled in Python, SQL, Excel, Tableau,
Power BI, data visualization, reporting, and machine learning.
"""

In [ ]:
clean_resume = preprocess_text(resume_text)
print(clean_resume)

In [ ]:
# 4. Transform resume using same TF-IDF vectorizer
resume_vector = tfidf_vectorizer.transform([clean_resume])

print(resume_vector.shape)

Compute cosine similarity

In [ ]:
# 5. Compute cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

tfidf_scores = cosine_similarity(resume_vector, tfidf_matrix).flatten()

print(tfidf_scores[:10])

In [ ]:
# 6. Get top 10 jobs
top_indices = tfidf_scores.argsort()[::-1][:10]

tfidf_recommendations = df.iloc[top_indices].copy()
tfidf_recommendations["tfidf_score"] = tfidf_scores[top_indices]

tfidf_recommendations[
    ["title", "company_name", "location", "formatted_experience_level", "tfidf_score"]
]

### 5a. SENTENCE BERT EMBEDDINGS

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
import pandas as pd

df = pd.read_csv("linkedin_jobs_preprocessed.csv")

print(df.shape)

In [ ]:
# 3 load sbert model
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# create job embeddings
job_texts = df["job_text"].fillna("").tolist()

job_embeddings = sbert_model.encode(
    job_texts,
    show_progress_bar=True,
    batch_size=32
)

print(job_embeddings.shape)

In [ ]:
# 5. Save embeddings
np.save("job_sbert_embeddings.npy", job_embeddings)

print("Saved job_sbert_embeddings.npy")

In [ ]:
resume_text = """
Experienced data analyst skilled in Python, SQL, Excel, Tableau,
Power BI, data visualization, reporting, dashboards, and machine learning.
"""

In [ ]:
resume_embedding = sbert_model.encode([resume_text])
print(resume_embedding.shape)

In [ ]:
# Compute SBERT similarity
sbert_scores = cosine_similarity(
    resume_embedding,
    job_embeddings
).flatten()

print(sbert_scores[:10])

In [ ]:

top_indices = sbert_scores.argsort()[::-1][:10]

sbert_recommendations = df.iloc[top_indices].copy()
sbert_recommendations["sbert_score"] = sbert_scores[top_indices]

sbert_recommendations[
    ["title", "company_name", "location", "formatted_experience_level", "sbert_score"]
]

## Fine Tune SBERT

In [ ]:
#step 5 fine tune sbert
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import pandas as pd
import random

In [ ]:
df = pd.read_csv("linkedin_jobs_preprocessed.csv")
df = df.dropna(subset=["job_text"]).reset_index(drop=True)

In [ ]:
def create_augmented_text(text):
    words = str(text).split()

    if len(words) <= 30:
        return str(text)

    keep_count = int(len(words) * 0.75)
    selected_words = random.sample(words, keep_count)

    return " ".join(selected_words)

In [ ]:
train_df = df.sample(n=3000, random_state=42)

train_examples = []

for text in train_df["job_text"]:
    original = str(text)
    augmented = create_augmented_text(original)

    train_examples.append(
        InputExample(texts=[original, augmented])
    )

print(len(train_examples))

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=16
)

train_loss = losses.MultipleNegativesRankingLoss(model)

In [ ]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=100,
    show_progress_bar=True
)

In [ ]:
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model.save(str(MODEL_DIR / "career_sbert_finetuned"))

print("Fine-tuned SBERT saved.")

### 5c. Generate fine tuned embeddings


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

sbert_model = SentenceTransformer("career_sbert_finetuned")

In [ ]:
job_texts = df["job_text"].fillna("").tolist()

job_embeddings_finetuned = sbert_model.encode(
    job_texts,
    show_progress_bar=True,
    batch_size=32
)

print(job_embeddings_finetuned.shape)

In [ ]:
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

np.save(
    ARTIFACTS_DIR / "job_sbert_finetuned_embeddings.npy",
    job_embeddings_finetuned
)

print("Saved fine-tuned job embeddings.")

## K MEANS CLUSTERING

In [ ]:
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
job_embeddings_norm = normalize(job_embeddings_finetuned)

In [ ]:
silhouette_scores = {}

for k in range(5, 16):
    kmeans_temp = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels_temp = kmeans_temp.fit_predict(job_embeddings_norm)

    score = silhouette_score(
        job_embeddings_norm,
        labels_temp,
        sample_size=2000,
        random_state=42
    )

    silhouette_scores[k] = score
    print(f"K={k}, Silhouette Score={score:.4f}")

In [ ]:
best_k = 8   # change this based on best silhouette score

kmeans_model = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans_model.fit_predict(job_embeddings_norm)

In [ ]:
df["cluster"].value_counts().sort_index()

In [ ]:
for cluster_id in sorted(df["cluster"].unique()):
    print(f"\nCluster {cluster_id}")
    print("-" * 50)
    print(
        df[df["cluster"] == cluster_id]["title"]
        .value_counts()
        .head(10)
    )

In [ ]:
import joblib
df.to_csv(
    PROCESSED_DIR / "linkedin_jobs_clustered.csv",
    index=False
)

joblib.dump(
    kmeans_model,
    MODEL_DIR / "kmeans_model.pkl"
)

print("Saved clustered dataset and K-Means model.")

In [ ]:
# Number of jobs in each cluster
print("Jobs per cluster:")
print(df["cluster"].value_counts().sort_index())

print("\n" + "="*80)

# Print the most common job titles in each cluster
for cluster_id in sorted(df["cluster"].unique()):

    print(f"\nCLUSTER {cluster_id}")
    print("-" * 50)

    top_titles = (
        df[df["cluster"] == cluster_id]["title"]
        .value_counts()
        .head(15)
    )

    print(top_titles)

    print("\n" + "="*80)

# HYBRID MODEL 
## TF-IDF baseline + fine-tuned SBERT embeddings + K-Means cluster filtering

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

In [ ]:
from pathlib import Path

BASE_DIR = Path(".")

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = BASE_DIR / "models"
ARTIFACTS_DIR = BASE_DIR / "artifacts"

In [ ]:
from pathlib import Path

BASE_DIR = Path(".")

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = BASE_DIR / "models"
ARTIFACTS_DIR = BASE_DIR / "artifacts"

In [ ]:
df = pd.read_csv(
    PROCESSED_DIR / "linkedin_jobs_clustered.csv"
)

sbert_model = SentenceTransformer(
    str(MODEL_DIR / "career_sbert_finetuned")
)

kmeans_model = joblib.load(
    MODEL_DIR / "kmeans_model.pkl"
)

job_embeddings = np.load(
    ARTIFACTS_DIR / "job_sbert_finetuned_embeddings.npy"
)

job_embeddings_norm = normalize(job_embeddings)

print("Dataset shape:", df.shape)
print("Embedding shape:", job_embeddings_norm.shape)

In [ ]:
#rebuild tfidf vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df["clean_job_text"])

In [ ]:
#Hybrid recommendation function
def recommend_jobs_hybrid(resume_text, top_n=10):
    # 1. Preprocess resume for TF-IDF
    clean_resume = preprocess_text(resume_text)

    # 2. TF-IDF similarity
    resume_tfidf = tfidf_vectorizer.transform([clean_resume])
    tfidf_scores = cosine_similarity(resume_tfidf, tfidf_matrix).flatten()

    # 3. Fine-tuned SBERT similarity
    resume_embedding = sbert_model.encode([resume_text])
    resume_embedding_norm = normalize(resume_embedding)

    sbert_scores = cosine_similarity(
        resume_embedding_norm,
        job_embeddings_norm
    ).flatten()

    # 4. Find nearest cluster
    resume_cluster = kmeans_model.predict(resume_embedding_norm)[0]

    # 5. Keep only jobs from same cluster
    cluster_indices = df[df["cluster"] == resume_cluster].index

    # 6. Hybrid score
    hybrid_scores = (0.4 * tfidf_scores) + (0.6 * sbert_scores)

    # 7. Rank only inside cluster
    cluster_scores = hybrid_scores[cluster_indices]

    top_cluster_positions = cluster_scores.argsort()[::-1][:top_n]
    top_indices = cluster_indices[top_cluster_positions]

    recommendations = df.loc[top_indices].copy()
    recommendations["tfidf_score"] = tfidf_scores[top_indices]
    recommendations["sbert_score"] = sbert_scores[top_indices]
    recommendations["hybrid_score"] = hybrid_scores[top_indices]
    recommendations["resume_cluster"] = resume_cluster

    return recommendations[
        [
            "title",
            "company_name",
            "location",
            "formatted_work_type",
            "formatted_experience_level",
            "normalized_salary",
            "tfidf_score",
            "sbert_score",
            "hybrid_score",
            "cluster",
            "job_posting_url"
        ]
    ]

In [ ]:
resume_text = """
Data analyst with experience in Python, SQL, Excel, Tableau, Power BI,
data visualization, dashboard development, reporting, and machine learning.
"""

recommend_jobs_hybrid(resume_text, top_n=10)

In [ ]:
# --------------------------------------------------
# EVALUATION METRICS
# --------------------------------------------------

def is_relevant_title(job_title, relevant_titles):
    job_title = str(job_title).lower()

    for relevant_title in relevant_titles:
        if relevant_title.lower() in job_title:
            return True

    return False


def precision_at_k(recommendations_df, relevant_titles, k=10):
    recommended_titles = recommendations_df["title"].head(k).tolist()

    hits = sum(
        1 for title in recommended_titles
        if is_relevant_title(title, relevant_titles)
    )

    return hits / k


def recall_at_k(recommendations_df, relevant_titles, k=10):
    recommended_titles = recommendations_df["title"].head(k).tolist()

    hits = sum(
        1 for title in recommended_titles
        if is_relevant_title(title, relevant_titles)
    )

    if len(relevant_titles) == 0:
        return 0

    return hits / len(relevant_titles)

Matched skills = skills in both resume and job

Missing skills = skills required by job but missing from resume

In [ ]:
#  create a skill dictionary 
skill_keywords = [
    "python", "java", "javascript", "sql", "excel", "tableau", "power bi",
    "machine learning", "deep learning", "nlp", "data analysis",
    "data visualization", "statistics", "pandas", "numpy", "scikit-learn",
    "tensorflow", "pytorch", "aws", "azure", "docker", "kubernetes",
    "react", "node.js", "html", "css", "git", "github",
    "communication", "leadership", "project management", "agile",
    "marketing", "seo", "sales", "recruitment", "hr", "finance"
]

In [ ]:
#  skill extraction function
def extract_skills(text, skill_list):
    text = str(text).lower()
    found_skills = []

    for skill in skill_list:
        if skill.lower() in text:
            found_skills.append(skill)

    return sorted(list(set(found_skills)))

In [ ]:
#  matched and msisng skill function
def get_skill_gap(resume_text, job_description):
    resume_skills = set(extract_skills(resume_text, skill_keywords))
    job_skills = set(extract_skills(job_description, skill_keywords))

    matched_skills = sorted(list(resume_skills.intersection(job_skills)))
    missing_skills = sorted(list(job_skills - resume_skills))

    return matched_skills, missing_skills

In [ ]:
#add skill gap to recommendations
def recommend_jobs_with_skill_gap(resume_text, top_n=10):
    recommendations = recommend_jobs_hybrid(resume_text, top_n)

    matched_list = []
    missing_list = []

    for idx in recommendations.index:
        job_description = df.loc[idx, "description"]

        matched_skills, missing_skills = get_skill_gap(
            resume_text,
            job_description
        )

        matched_list.append(", ".join(matched_skills))
        missing_list.append(", ".join(missing_skills))

    recommendations["matched_skills"] = matched_list
    recommendations["missing_skills"] = missing_list

    return recommendations

In [ ]:
# test
resume_text = """
Data analyst with experience in Python, SQL, Excel, Tableau, Power BI,
data visualization, dashboard development, reporting, statistics, and machine learning.
"""

recommend_jobs_with_skill_gap(resume_text, top_n=10)

In [ ]:
# Evaluattion
test_resumes = {
    "Data Analyst": """
    Data Analyst with experience in Python, SQL, Excel, Tableau,
    Power BI, dashboard reporting, data visualization, statistics,
    and machine learning.
    """,

    "Software Engineer": """
    Software Engineer skilled in Java, Python, REST APIs, Docker,
    AWS, Git, backend development, databases, and cloud deployment.
    """,

    "HR Recruiter": """
    HR professional with experience in recruitment, talent acquisition,
    interviewing, onboarding, employee relations, and communication.
    """,

    "Digital Marketing": """
    Digital marketer experienced in SEO, Google Analytics, social media,
    content marketing, campaign management, and sales strategy.
    """
}

In [ ]:
def recommend_jobs_tfidf(resume_text, top_n=5):
    clean_resume = preprocess_text(resume_text)
    resume_vector = tfidf_vectorizer.transform([clean_resume])

    scores = cosine_similarity(resume_vector, tfidf_matrix).flatten()

    top_indices = scores.argsort()[::-1][:top_n]

    results = df.iloc[top_indices].copy()
    results["tfidf_score"] = scores[top_indices]

    return results[["title", "company_name", "location", "tfidf_score"]]

In [ ]:
def recommend_jobs_sbert(resume_text, top_n=5):
    resume_embedding = sbert_model.encode([resume_text])
    resume_embedding_norm = normalize(resume_embedding)

    scores = cosine_similarity(
        resume_embedding_norm,
        job_embeddings_norm
    ).flatten()

    top_indices = scores.argsort()[::-1][:top_n]

    results = df.iloc[top_indices].copy()
    results["sbert_score"] = scores[top_indices]

    return results[["title", "company_name", "location", "sbert_score"]]

In [ ]:
for resume_name, resume_text in test_resumes.items():
    print("\n" + "="*80)
    print(f"TEST RESUME: {resume_name}")
    print("="*80)

    print("\nTF-IDF Recommendations:")
    display(recommend_jobs_tfidf(resume_text, top_n=5))

    print("\nSBERT Recommendations:")
    display(recommend_jobs_sbert(resume_text, top_n=5))

    print("\nHybrid Recommendations:")
    display(recommend_jobs_hybrid(resume_text, top_n=5))